In [ ]:
import pandas as pd
import numpy as np
import torch
from functools import reduce
import warnings

# 設定環境
warnings.filterwarnings('ignore')
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ==========================================
# 1. 讀取與合併資料 (Load & Merge)
# ==========================================
print("步驟 1/3: 正在讀取並合併檔案...")

base_path = "C:\\Users\\user\\Desktop\\HW-NCHU\\meeting\\GPR\\data\\"

# 定義檔案路徑
file_paths = {
    "PM25":   base_path + "PM2.5_FPCA_2025.csv", # Input Feature 1 & Train Target
    "WIND_U": base_path + "WIND_U_FPCA_2025.csv",     # Input Feature 2
    "WIND_V": base_path + "WIND_V_FPCA_2025.csv",     # Input Feature 3
    "RH":     base_path + "RH_FPCA_2025.csv",    # Input Feature 4
    "TEMP":   base_path + "AMB_TEMP_FPCA_2025.csv"     # Input Feature 5
}

# --- A. 處理 5 個 FPCA 特徵 ---
dfs_features = []
for feature_name, path in file_paths.items():
    df = pd.read_csv(path)
    # 移除多餘欄位，只留時間跟站點數據
    cols = [c for c in df.columns if c not in ['date', 'Time', 'year', 'SubjectID']]
    df = df[cols]
    df['PublishTime'] = pd.to_datetime(df['PublishTime'])
    
    # 轉成長表格 (Long Format): [PublishTime, Station, Feature_Value]
    df_melt = df.melt(id_vars=['PublishTime'], var_name='Station', value_name=feature_name)
    dfs_features.append(df_melt)

# 合併這 5 個表格 (Inner Join)
df_merged_features = reduce(lambda left, right: pd.merge(left, right, on=['PublishTime', 'Station'], how='inner'), dfs_features)

# --- B. 處理 Raw Data (用於 Test 驗證) ---
raw_df = pd.read_csv(base_path + "PM2.5.csv")
raw_df['PublishTime'] = pd.to_datetime(raw_df['PublishTime'])
# 轉長表格
raw_melted = raw_df.melt(id_vars=['PublishTime'], var_name='Station', value_name='PM25_Raw')

# --- C. 最終合併 ---
# 把 Raw Data 合併進去 (Left Join)
df_all = pd.merge(df_merged_features, raw_melted, on=['PublishTime', 'Station'], how='left')

# 確保欄位順序固定: [PM25, U, V, RH, TEMP, PM25_Raw]
# Index 對照: 0=PM25, 1=U, 2=V, 3=RH, 4=TEMP, 5=Raw
cols_order = ['PublishTime', 'Station', 'PM25', 'WIND_U', 'WIND_V', 'RH', 'TEMP', 'PM25_Raw']
df_all = df_all[cols_order]

# 排序
df_all = df_all.sort_values(by=['Station', 'PublishTime'])

print("合併完成！資料預覽：")
print(df_all.head())

# ==========================================
# 2. 滑動視窗切分 (Sliding Window) - 已修改為不重疊
# ==========================================
print("\n步驟 2/3: 正在切分時間視窗 (每 24 小時跳一次)...")

# 參數設定
INPUT_HOURS = 24
OUTPUT_HOURS = 72
TOTAL_WINDOW = INPUT_HOURS + OUTPUT_HOURS # 48
STEP_SIZE = 24  

SPLIT_DATE = pd.Timestamp('2025-01-01')
TRAIN_START = pd.Timestamp('2018-01-01')
TEST_END = pd.Timestamp('2025-11-30')

# 準備存放結果的字典
station_datasets = {}

# 取得所有站點清單
stations = df_all['Station'].unique()

for station in stations:
    # 取出單一站點資料
    df_station = df_all[df_all['Station'] == station].copy()
    df_station = df_station.set_index('PublishTime').sort_index()
    
    # 強制補齊時間軸
    # 加上內層的中括號
    df_station = df_station[['PM25', 'WIND_U', 'WIND_V', 'RH', 'TEMP', 'PM25_Raw']].asfreq('H')
    
    # 轉成 Numpy Array 加速
    data_values = df_station.values  # Shape: (T, 6)
    times = df_station.index
    
    # 暫存 list
    train_X_list, train_Y_list = [], []
    test_X_list, test_Y_raw_list = [], []
    
    num_samples = len(data_values) - TOTAL_WINDOW + 1
    
    if num_samples <= 0:
        continue

    # ============================================================
    # 【這裡改了】加上 step (第三個參數)
    # range(start, stop, step)
    # i 會變成: 0, 24, 48, 72... 
    # 這樣視窗就不會重疊：
    # i=0:  0~23(Input) -> 24~47(Output)
    # i=24: 24~47(Input) -> 48~71(Output)
    # ============================================================
    for i in range(0, num_samples, STEP_SIZE):
        
        window = data_values[i : i + TOTAL_WINDOW]
        current_time = times[i]
        
        # --- 1. 處理 Input (前 24 小時) ---
        x_window = window[:INPUT_HOURS, 0:5]
        
        # 如果 Input 有任何 NaN，跳過 (這一天資料不全，不預測)
        if np.isnan(x_window).any():
            continue
            
        # --- 2. 處理 Output (後 24 小時) ---
        if TRAIN_START <= current_time < SPLIT_DATE:
            # === Training Set ===
            y_window_smooth = window[INPUT_HOURS:, 0] # 這裡會自動切出 72 小時
            
            if np.isnan(y_window_smooth).any():
                continue
            
            train_X_list.append(x_window)
            train_Y_list.append(y_window_smooth)
            
        elif SPLIT_DATE <= current_time <= (TEST_END - pd.Timedelta(hours=TOTAL_WINDOW)):
            # === Testing Set ===
            y_window_raw = window[INPUT_HOURS:, 5] # 這裡會自動切出 72 小時
            
            if np.isnan(y_window_raw).all():
                continue
                
            test_X_list.append(x_window)
            test_Y_raw_list.append(y_window_raw)
    
    # 存入字典
    if len(train_X_list) > 0 and len(test_X_list) > 0:
        station_datasets[station] = {
            'train_x': torch.tensor(np.array(train_X_list), dtype=torch.float32).to(device),
            'train_y': torch.tensor(np.array(train_Y_list), dtype=torch.float32).to(device),
            'test_x':  torch.tensor(np.array(test_X_list), dtype=torch.float32).to(device),
            'test_y_raw': np.array(test_Y_raw_list) 
        }

print(f"切分完成！共有 {len(station_datasets)} 個站點有有效資料。")


# ==========================================
# 3. 檢查結果 (Verification)
# ==========================================
print("\n步驟 3/3: 檢查所有站點的資料形狀...")

if len(station_datasets) > 0:
    print(f"{'='*60}")
    # 使用 items() 同時取得 站點名稱(name) 和 資料(data)
    for station_name, data in station_datasets.items():
        print(f"站點: {station_name}")
        print(f"  Train Input Shape: {data['train_x'].shape}")
        print(f"  Train Label Shape: {data['train_y'].shape}")
        print(f"  Test  Input Shape: {data['test_x'].shape}")
        print(f"  Test  Label Shape: {data['test_y_raw'].shape} (含 NaN)")
        print(f"{'-'*30}") # 分隔線，讓閱讀更清楚
        
    print(f"\n檢查完畢！共 {len(station_datasets)} 個站點。")
else:
    print("警告：沒有產生任何有效資料。")

Using device: cuda
步驟 1/3: 正在讀取並合併檔案...
合併完成！資料預覽：
              PublishTime Station       PM25    WIND_U    WIND_V         RH  \
69384 2018-01-01 00:00:00      三義  20.879745 -1.431222 -3.516175  92.205352   
69385 2018-01-01 01:00:00      三義  20.760245 -1.426244 -3.463227  92.892279   
69386 2018-01-01 02:00:00      三義  20.131393 -1.416487 -3.410281  93.741596   
69387 2018-01-01 03:00:00      三義  19.218529 -1.393757 -3.342953  94.718673   
69388 2018-01-01 04:00:00      三義  18.036123 -1.370112 -3.299227  95.821793   

            TEMP  PM25_Raw  
69384  11.606834      20.5  
69385  11.513281      21.5  
69386  11.411473      19.5  
69387  11.324640      21.5  
69388  11.233603      19.5  

步驟 2/3: 正在切分時間視窗 (每 24 小時跳一次)...
切分完成！共有 74 個站點有有效資料。

步驟 3/3: 檢查所有站點的資料形狀...
站點: 三義
  Train Input Shape: torch.Size([2367, 24, 5])
  Train Label Shape: torch.Size([2367, 72])
  Test  Input Shape: torch.Size([326, 24, 5])
  Test  Label Shape: (326, 72) (含 NaN)
------------------------------
站點: 中壢


# deeponet

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import gc
from sklearn.metrics import mean_absolute_error, mean_squared_error

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =============================================================================
# 模型定義
# =============================================================================
class DeepONet(nn.Module):
    def __init__(self, input_dim=24, p=128):
        """
        input_dim: 24 (小時) * 5 (特徵) = 120
        p: Branch 和 Trunk 輸出的壓縮維度
        """
        super().__init__()

        self.branch = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, p),
        )

        self.trunk = nn.Sequential(
            nn.Linear(1, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, p),
        )

        self.bias = nn.Parameter(torch.zeros(1))
    
    def forward(self, branch_input, trunk_input):
        # branch_input shape: [Batch, 24]
        # trunk_input shape: [72, 1]
        
        b = self.branch(branch_input)     # [Batch, p]
        t = self.trunk(trunk_input)      # [72, p]
        
        # 利用矩陣乘法計算點積: [Batch, p] @ [p, 72] -> [Batch, 72]
        return torch.matmul(b, t.T) + self.bias


# =============================================================================
# 訓練參數與資料準備
# =============================================================================
P = 128
LR = 0.001
TRAINING_ITER = 1000
WEIGHT_DECAY = 1e-2

# 根據你的標籤長度 72，定義 Trunk 的輸入時間點 (例如將 72 小時映射到 0~1)
trunk_input = torch.linspace(0, 1, 72).unsqueeze(-1).to(device)

print(f"\n{'='*10} DeepONet 訓練開始 {'='*10}")
results_list = []

# 假設 station_datasets 已經存在
for i, (station_name, dataset) in enumerate(station_datasets.items()):
    train_x = dataset['train_x'][:, :, :1].to(torch.float32).to(device)
    train_x_flat = train_x.reshape(train_x.size(0), -1) 
    
    train_y = dataset['train_y'].to(torch.float32).to(device) 
    
    test_x = dataset['test_x'][:, :, :1].to(torch.float32).to(device)
    test_x_flat = test_x.reshape(test_x.size(0), -1)
    
    test_y_raw = dataset['test_y_raw'] # 用於評估的原始值 (含 NaN)

    model = DeepONet(input_dim=24, p=P).to(device)
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=TRAINING_ITER, eta_min=1e-5
    )
    criterion = nn.MSELoss()
    
    # --- 訓練階段 ---
    model.train()
    for epoch in range(TRAINING_ITER):
        optimizer.zero_grad()
        
        # Forward: 輸入平坦化的 x 和定義好的 trunk_t
        preds = model(train_x_flat, trunk_input) # 輸出 [2535, 72]
        
        loss = criterion(preds, train_y)
        loss.backward()
        optimizer.step()
        scheduler.step()
        '''
        if (epoch + 1) % 100 == 0:
             print(f"Station: {station_name} | Epoch [{epoch+1}/{TRAINING_ITER}] | Loss: {loss.item():.6f}")
        '''
    # --- 評估階段 ---
    model.eval()
    with torch.no_grad():
        y_pred = model(test_x_flat, trunk_input).cpu().numpy()
    
    # 處理標籤中的 NaN 並計算指標
    mask = ~np.isnan(test_y_raw)
    if mask.sum() > 0:
        mae = mean_absolute_error(test_y_raw[mask], y_pred[mask])
        rmse = np.sqrt(mean_squared_error(test_y_raw[mask], y_pred[mask]))
        results_list.append({'Station': station_name, 'MAE': mae, 'RMSE': rmse})
        print(f"[{i+1}/{len(station_datasets)}] {station_name:<5} | MAE: {mae:.4f} | RMSE: {rmse:.4f}")
    
    # 釋放記憶體
    del model, optimizer, scheduler, y_pred, train_x, train_x_flat, test_x, test_x_flat
    gc.collect()
    torch.cuda.empty_cache()

# --- 輸出最終結果 ---
if results_list:
    results_df = pd.DataFrame(results_list)
    print(f"\n{'='*20} 最終結果 {'='*20}")
    print(f"平均 MAE : {results_df['MAE'].mean():.4f}")
    print(f"平均 RMSE: {results_df['RMSE'].mean():.4f}")


========== DeepONet 訓練開始 ==========
[1/74] 三義    | MAE: 5.1525 | RMSE: 6.6803
[2/74] 中壢    | MAE: 5.7540 | RMSE: 7.9051
[3/74] 中山    | MAE: 5.3372 | RMSE: 7.3086
[4/74] 二林    | MAE: 7.2358 | RMSE: 9.8449
[5/74] 仁武    | MAE: 6.2267 | RMSE: 8.5864
[6/74] 冬山    | MAE: 3.7548 | RMSE: 5.2857
[7/74] 前金    | MAE: 5.9736 | RMSE: 8.2902
[8/74] 前鎮    | MAE: 6.1244 | RMSE: 8.4026
[9/74] 南投    | MAE: 6.1397 | RMSE: 8.2669
[10/74] 古亭    | MAE: 5.4041 | RMSE: 7.3264
[11/74] 善化    | MAE: 6.0603 | RMSE: 8.3355
[12/74] 嘉義    | MAE: 6.6704 | RMSE: 9.0512
[13/74] 土城    | MAE: 5.5738 | RMSE: 7.6530
[14/74] 埔里    | MAE: 5.4680 | RMSE: 7.2371
[15/74] 基隆    | MAE: 4.9909 | RMSE: 6.8492
[16/74] 士林    | MAE: 5.2923 | RMSE: 7.1016
[17/74] 大園    | MAE: 6.0376 | RMSE: 8.3107
[18/74] 大寮    | MAE: 6.7424 | RMSE: 9.1670
[19/74] 大里    | MAE: 6.2328 | RMSE: 8.4342
[20/74] 安南    | MAE: 6.2340 | RMSE: 8.5437
[21/74] 宜蘭    | MAE: 3.9442 | RMSE: 5.4694
[22/74] 富貴角   | MAE: 5.1460 | RMSE: 7.2422
[23/74] 小港    | MAE: 6.141

FEDONET


In [23]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import gc
import math
from sklearn.metrics import mean_absolute_error, mean_squared_error

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =============================================================================
# Learnable Fourier Features
# 參考 Tancik et al. 2020，但 B 從固定改為可學
# γ(y) = [sin(2π B y), cos(2π B y)]
# B ∈ ℝ^{m×1}, 初始化 N(0, 1), 跟著反向傳播訓練
# =============================================================================
class LearnableFourierFeatures(nn.Module):
    def __init__(self, input_dim=1, num_frequencies=32, sigma=1.0):
        """
        input_dim       : trunk 輸入維度 (你的 case 是 1, 因為 y 是 normalized 未來小時)
        num_frequencies : m, 輸出維度會是 2m (sin/cos 各 m)
        sigma           : N(0, sigma) 初始化的標準差 (預設 1.0, 你說的 normal(0,1))
        """
        super().__init__()
        # B: shape (num_frequencies, input_dim)
        # 用 nn.Parameter, 自動參與 backprop
        self.B = nn.Parameter(torch.randn(num_frequencies, input_dim) * sigma)

    def forward(self, y):
        # y: (M, input_dim)  e.g. (72, 1)
        # y @ B^T: (M, num_frequencies)
        proj = 2 * math.pi * y @ self.B.T  # (M, num_frequencies)
        # 拼起來: (M, 2 * num_frequencies)
        return torch.cat([torch.sin(proj), torch.cos(proj)], dim=-1)


# =============================================================================
# 模型定義 — Vanilla DeepONet + Learnable Fourier Trunk
# =============================================================================
class DeepONet(nn.Module):
    def __init__(self, input_dim=24, p=128, num_frequencies=32):
        super().__init__()

        # Branch 完全不動
        self.branch = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, p),
        )

        # Trunk: 加上 Fourier embedding 在最前面
        # 原本第一層是 Linear(1, 128), 現在變成 Linear(2*num_frequencies, 128)
        self.fourier = LearnableFourierFeatures(
            input_dim=1, num_frequencies=num_frequencies, sigma=1.0
        )
        self.trunk = nn.Sequential(
            nn.Linear(2 * num_frequencies, 128),  # ← 唯一改的地方: 1 → 2m
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, p),
        )

        self.bias = nn.Parameter(torch.zeros(1))

    def forward(self, branch_input, trunk_input):
        # branch_input shape: [Batch, 24]
        # trunk_input shape: [72, 1]

        b = self.branch(branch_input)              # [Batch, p]

        # Trunk: 先過 Fourier embedding 再進 MLP
        y_embed = self.fourier(trunk_input)        # [72, 2*num_frequencies]
        t = self.trunk(y_embed)                    # [72, p]

        return torch.matmul(b, t.T) + self.bias    # [Batch, 72]


# =============================================================================
# 訓練參數與資料準備（跟你 vanilla 版完全一致）
# =============================================================================
P = 128
LR = 0.001
TRAINING_ITER = 1500
WEIGHT_DECAY = 1e-2
NUM_FREQUENCIES = 32  # ← 新增超參數, 之後可以調 (16 / 32 / 64)

trunk_input = torch.linspace(0, 1, 72).unsqueeze(-1).to(device)

print(f"\n{'='*10} DeepONet + Learnable Fourier Trunk 訓練開始 {'='*10}")
print(f"Fourier num_frequencies = {NUM_FREQUENCIES} (output dim = {2*NUM_FREQUENCIES})")
results_list = []

for i, (station_name, dataset) in enumerate(station_datasets.items()):
    train_x = dataset['train_x'][:, :, :1].to(torch.float32).to(device)
    train_x_flat = train_x.reshape(train_x.size(0), -1)

    train_y = dataset['train_y'].to(torch.float32).to(device)

    test_x = dataset['test_x'][:, :, :1].to(torch.float32).to(device)
    test_x_flat = test_x.reshape(test_x.size(0), -1)

    test_y_raw = dataset['test_y_raw']

    model = DeepONet(input_dim=24, p=P, num_frequencies=NUM_FREQUENCIES).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=TRAINING_ITER, eta_min=1e-5
    )
    criterion = nn.MSELoss()

    model.train()
    for epoch in range(TRAINING_ITER):
        optimizer.zero_grad()
        preds = model(train_x_flat, trunk_input)
        loss = criterion(preds, train_y)
        loss.backward()
        optimizer.step()
        scheduler.step()

    model.eval()
    with torch.no_grad():
        y_pred = model(test_x_flat, trunk_input).cpu().numpy()

    mask = ~np.isnan(test_y_raw)
    if mask.sum() > 0:
        mae = mean_absolute_error(test_y_raw[mask], y_pred[mask])
        rmse = np.sqrt(mean_squared_error(test_y_raw[mask], y_pred[mask]))
        results_list.append({'Station': station_name, 'MAE': mae, 'RMSE': rmse})
        print(f"[{i+1}/{len(station_datasets)}] {station_name:<5} | MAE: {mae:.4f} | RMSE: {rmse:.4f}")

    del model, optimizer, scheduler, y_pred, train_x, train_x_flat, test_x, test_x_flat
    gc.collect()
    torch.cuda.empty_cache()

if results_list:
    results_df = pd.DataFrame(results_list)
    print(f"\n{'='*20} 最終結果 {'='*20}")
    print(f"平均 MAE : {results_df['MAE'].mean():.4f}")
    print(f"平均 RMSE: {results_df['RMSE'].mean():.4f}")


========== DeepONet + Learnable Fourier Trunk 訓練開始 ==========
Fourier num_frequencies = 32 (output dim = 64)
[1/74] 三義    | MAE: 5.3244 | RMSE: 6.9866
[2/74] 中壢    | MAE: 5.9010 | RMSE: 8.0314
[3/74] 中山    | MAE: 5.4256 | RMSE: 7.4768
[4/74] 二林    | MAE: 7.2067 | RMSE: 9.8057
[5/74] 仁武    | MAE: 6.2858 | RMSE: 8.5979
[6/74] 冬山    | MAE: 3.7787 | RMSE: 5.3391
[7/74] 前金    | MAE: 6.0092 | RMSE: 8.3358
[8/74] 前鎮    | MAE: 6.1747 | RMSE: 8.4870
[9/74] 南投    | MAE: 6.1539 | RMSE: 8.3528
[10/74] 古亭    | MAE: 5.3749 | RMSE: 7.3405
[11/74] 善化    | MAE: 6.1393 | RMSE: 8.3888
[12/74] 嘉義    | MAE: 6.8721 | RMSE: 9.3052
[13/74] 土城    | MAE: 5.7345 | RMSE: 7.8791
[14/74] 埔里    | MAE: 5.4275 | RMSE: 7.2716
[15/74] 基隆    | MAE: 5.0314 | RMSE: 6.8848
[16/74] 士林    | MAE: 5.3187 | RMSE: 7.1148
[17/74] 大園    | MAE: 6.1873 | RMSE: 8.5478
[18/74] 大寮    | MAE: 6.7005 | RMSE: 9.1536
[19/74] 大里    | MAE: 6.3033 | RMSE: 8.5481
[20/74] 安南    | MAE: 6.2709 | RMSE: 8.7036
[21/74] 宜蘭    | MAE: 4.0504 | RMSE: 5.6

pod

In [29]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import gc
from sklearn.metrics import mean_absolute_error, mean_squared_error

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =============================================================================
# POD basis computation (per station, via GPU SVD)
# =============================================================================
def compute_pod_basis(train_y, p):
    """
    train_y : (N, 72) torch tensor on device, float32
    p       : number of POD basis to keep (latent dim)

    Returns:
        mean_y : (72,)   train_y 的平均 (作為 bias function)
        basis  : (72, p) POD basis matrix, 每個 column 是一個 basis function
                 等於 train_y - mean 的右奇異向量
    """
    # 步驟 1: 中心化 (POD 必須先減 mean)
    mean_y = train_y.mean(dim=0)                       # (72,)
    Y_centered = train_y - mean_y.unsqueeze(0)         # (N, 72)

    # 步驟 2: SVD on GPU
    # ⚠️ 注意：torch.linalg.svd 在 fp32 對 ill-conditioned matrix 可能不穩
    # 為了穩定，先轉 fp64 算完再轉回去
    Y64 = Y_centered.to(torch.float64)
    U, S, Vh = torch.linalg.svd(Y64, full_matrices=False)
    # U:  (N, min(N,72))
    # S:  (min(N,72),)
    # Vh: (min(N,72), 72)

    # 步驟 3: 取前 p 個 basis
    # Vh 的每一 row 是一個右奇異向量, 我們要 column-orient → 取 Vh[:p].T
    basis = Vh[:p].T.contiguous()                      # (72, p)

    return mean_y.to(torch.float32), basis.to(torch.float32)


# =============================================================================
# POD-DeepONet
# Branch 跟 vanilla 完全一樣
# Trunk: 移除 NN, 改用固定 POD basis + mean function 作為 bias
# =============================================================================
class PODDeepONet(nn.Module):
    def __init__(self, input_dim=24, p=32):
        super().__init__()

        # Branch: 跟 vanilla 完全一樣
        self.branch = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, p),
        )

        # POD basis 和 mean function 在外面算好後用 register_buffer 灌進來
        # 用 register_buffer 而不是 nn.Parameter, 表示「固定不訓練」
        self.register_buffer('basis', torch.zeros(72, p))   # trunk 取代品
        self.register_buffer('mean_y', torch.zeros(72))     # bias function

        # 注意：原 vanilla 的 scalar bias 移除了, 因為 mean_y 已經是 bias function

    def set_pod(self, mean_y, basis):
        """訓練前先 call 這個 method 把 POD 結果灌進來"""
        self.basis.copy_(basis)
        self.mean_y.copy_(mean_y)

    def forward(self, branch_input):
        # branch_input : (Batch, 24)
        b = self.branch(branch_input)                   # (Batch, p)
        # basis: (72, p), b: (Batch, p)
        # 想要 (Batch, 72): b @ basis.T
        out = b @ self.basis.T + self.mean_y           # (Batch, 72), broadcast mean_y
        return out


# =============================================================================
# 訓練參數
# =============================================================================
P = 32                          # ← latent dim 從 128 降到 32
LR = 0.001
TRAINING_ITER = 1000
WEIGHT_DECAY = 1e-2

print(f"\n{'='*10} POD-DeepONet 訓練開始 (p={P}) {'='*10}")
results_list = []
pod_info_list = []   # 記錄每站 POD 解釋變異量

for i, (station_name, dataset) in enumerate(station_datasets.items()):
    train_x = dataset['train_x'][:, :, :1].to(torch.float32).to(device)
    train_x_flat = train_x.reshape(train_x.size(0), -1)

    train_y = dataset['train_y'].to(torch.float32).to(device)

    test_x = dataset['test_x'][:, :, :1].to(torch.float32).to(device)
    test_x_flat = test_x.reshape(test_x.size(0), -1)
    test_y_raw = dataset['test_y_raw']

    # ============================================================
    # 【關鍵】per-station POD: 每站算自己的 basis
    # ============================================================
    mean_y, basis = compute_pod_basis(train_y, p=P)

    # 順便算 explained variance ratio, 給你看每站 p=32 抓到多少訊息
    Y_centered = train_y - mean_y.unsqueeze(0)
    total_var = (Y_centered ** 2).sum().item()
    reconstructed = (Y_centered @ basis) @ basis.T   # (N, 72)
    residual_var = ((Y_centered - reconstructed) ** 2).sum().item()
    explained_ratio = 1.0 - residual_var / total_var
    pod_info_list.append({'Station': station_name, 'ExplainedVar': explained_ratio})

    # ============================================================
    # Model + 灌入 POD
    # ============================================================
    model = PODDeepONet(input_dim=24, p=P).to(device)
    model.set_pod(mean_y, basis)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=TRAINING_ITER, eta_min=1e-5
    )
    criterion = nn.MSELoss()

    model.train()
    for epoch in range(TRAINING_ITER):
        optimizer.zero_grad()
        preds = model(train_x_flat)                     # ← 不再需要 trunk_input
        loss = criterion(preds, train_y)
        loss.backward()
        optimizer.step()
        scheduler.step()

    model.eval()
    with torch.no_grad():
        y_pred = model(test_x_flat).cpu().numpy()

    mask = ~np.isnan(test_y_raw)
    if mask.sum() > 0:
        mae = mean_absolute_error(test_y_raw[mask], y_pred[mask])
        rmse = np.sqrt(mean_squared_error(test_y_raw[mask], y_pred[mask]))
        results_list.append({'Station': station_name, 'MAE': mae, 'RMSE': rmse})
        print(f"[{i+1}/{len(station_datasets)}] {station_name:<5} | "
              f"MAE: {mae:.4f} | RMSE: {rmse:.4f} | "
              f"POD覆蓋: {explained_ratio*100:.2f}%")

    del model, optimizer, scheduler, y_pred
    del train_x, train_x_flat, test_x, test_x_flat, mean_y, basis
    gc.collect()
    torch.cuda.empty_cache()

if results_list:
    results_df = pd.DataFrame(results_list)
    pod_df = pd.DataFrame(pod_info_list)
    print(f"\n{'='*20} 最終結果 {'='*20}")
    print(f"平均 MAE : {results_df['MAE'].mean():.4f}")
    print(f"平均 RMSE: {results_df['RMSE'].mean():.4f}")
    print(f"平均 POD explained variance (p={P}): {pod_df['ExplainedVar'].mean()*100:.2f}%")
    print(f"  最低: {pod_df['ExplainedVar'].min()*100:.2f}% ({pod_df.loc[pod_df['ExplainedVar'].idxmin(), 'Station']})")
    print(f"  最高: {pod_df['ExplainedVar'].max()*100:.2f}% ({pod_df.loc[pod_df['ExplainedVar'].idxmax(), 'Station']})")


========== POD-DeepONet 訓練開始 (p=32) ==========
[1/74] 三義    | MAE: 5.2742 | RMSE: 6.8215 | POD覆蓋: 100.00%
[2/74] 中壢    | MAE: 5.8782 | RMSE: 7.9193 | POD覆蓋: 100.00%
[3/74] 中山    | MAE: 5.3611 | RMSE: 7.2819 | POD覆蓋: 100.00%
[4/74] 二林    | MAE: 7.3781 | RMSE: 9.8879 | POD覆蓋: 100.00%
[5/74] 仁武    | MAE: 6.3701 | RMSE: 8.6231 | POD覆蓋: 100.00%
[6/74] 冬山    | MAE: 3.7860 | RMSE: 5.2883 | POD覆蓋: 100.00%
[7/74] 前金    | MAE: 6.1363 | RMSE: 8.4085 | POD覆蓋: 100.00%
[8/74] 前鎮    | MAE: 6.1514 | RMSE: 8.3647 | POD覆蓋: 100.00%
[9/74] 南投    | MAE: 6.2576 | RMSE: 8.3965 | POD覆蓋: 100.00%
[10/74] 古亭    | MAE: 5.3621 | RMSE: 7.2441 | POD覆蓋: 100.00%
[11/74] 善化    | MAE: 6.2133 | RMSE: 8.3558 | POD覆蓋: 100.00%
[12/74] 嘉義    | MAE: 6.9323 | RMSE: 9.2578 | POD覆蓋: 100.00%
[13/74] 土城    | MAE: 5.5724 | RMSE: 7.5545 | POD覆蓋: 100.00%
[14/74] 埔里    | MAE: 5.4603 | RMSE: 7.2582 | POD覆蓋: 100.00%
[15/74] 基隆    | MAE: 4.9777 | RMSE: 6.7139 | POD覆蓋: 100.00%
[16/74] 士林    | MAE: 5.3585 | RMSE: 7.1099 | POD覆蓋: 100.00%
[